# LLM Fine-Tuning (Colab Free Tier)

## Task Description
Fine-tune a Large Language Model (LLM) on a custom dataset using Parameter-Efficient Fine-Tuning (PEFT) and Quantization (QLoRA).

**Focus:** This notebook is optimized for **Google Colab Free Tier** (T4 GPU, 16GB VRAM, 12GB RAM). We will use 4-bit quantization to fit the model in memory.

In this notebook, we will:
1. **Install & Import:** Set up `transformers`, `peft`, `bitsandbytes`, and `trl`.
2. **Data Preparation:** Load and **clean/format** a dataset (Guanaco-1k) for instruction tuning.
3. **Model Loading:** Load an open-source model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) in 4-bit precision.
4. **Fine-Tuning:** Configure LoRA (Low-Rank Adaptation) and train using `SFTTrainer`.
5. **Save:** Save the fine-tuned adapter weights.
6. **Inference:** Load the fine-tuned model and generate text to verify performance.


## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier.

## 1. Installation
We need `bitsandbytes` for 4-bit quantization, `peft` for LoRA, and `trl` for the Supervised Fine-Tuning (SFT) Trainer.

In [ ]:
!pip install matplotlib -q transformers torch peft bitsandbytes trl accelerate datasets

## 2. Imports and Setup
Import necessary libraries and check for GPU availability.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Load Data
Load raw data.

## 4. Data Cleaning
Format data.

In [ ]:
# Load Dataset
dataset_name = "mlabonne/guanaco-llama2-1k"
dataset = load_dataset(dataset_name, split="train")

print("Raw Data Example:")
print(dataset[0])

# Feature Engineering / Formatting Function
# We strictly format the text to ensure the model learns the instruction structure.
def format_instruction(sample):
    return f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"

# Note: The dataset 'guanaco-llama2-1k' actually uses a specific 'text' field already formatted in some versions,
# but for demonstration, let's assume we are re-formatting or the dataset has 'instruction'/'output' columns.
# Let's inspect the column names first.
print(f"\nDataset Columns: {dataset.column_names}")

# We will apply our formatting function to create a new 'formatted_text' column.
# This demonstrates how to process raw inputs into the model's expected format.
dataset = dataset.map(lambda x: {"formatted_text": format_instruction(x)})

# Cleaning Step: Remove samples that are too long to ensure efficient training
def filter_long_sequences(sample):
    return len(sample['formatted_text']) < 2048

original_len = len(dataset)
dataset = dataset.filter(filter_long_sequences)
print(f"\nFiltered dataset from {original_len} to {len(dataset)} samples.")

## 5. Load Model


In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded in 4-bit precision.")

## 6. Configure LoRA


In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8, # Rank
    bias="none",
    task_type="CAUSAL_LM",
)

## 7. Inference (Training)


In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False, # Use True if on A100, but False for T4 usually safe (or True if supported)
    max_grad_norm=0.3,
    max_steps=50, # Limit steps for demonstration speed. Remove this for full training.
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="formatted_text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

# Start Training
trainer.train()

## 8. Save Results (Model)


In [ ]:
new_model_name = "tinyllama-guanaco-adapter"
trainer.model.save_pretrained(new_model_name)
print(f"Adapter saved to {new_model_name}")

## 9. Advanced Usage (Inference)


In [ ]:
# Prompt
prompt = "What is the capital of France?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=100)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

## 10. Save Metrics
Save logs.

In [ ]:
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'frames' in locals():
        f.write("Video generated.")
    elif 'image' in locals():
        f.write("Image generated.")
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print('Results saved to results.txt')


In [ ]:
# Example: Load a different model
# model_name = "..."
# classifier = pipeline(..., model=model_name)
# results = classifier(your_input)
print("Try modifying the code above to use a different model!")